In [ ]:
# @title Upload 5 Berkas Excel Baru (1 Pembobot + 4 Bidang Dataset)
from google.colab import files
import os

print("Silakan unggah 5 file Excel Anda:")
print("1. Berkas Pembobot (W8)")
print("2. Berkas Dataset TPT")
print("3. Berkas Dataset Kesehatan")
print("4. Berkas Dataset Pendidikan")
print("5. Berkas Dataset Pertanian") #Tidak available untuk dataset 10 th
uploaded_files = files.upload()

# Menampilkan daftar berkas yang berhasil diunggah
print("\nBerkas yang berhasil diunggah:")
for filename in uploaded_files.keys():
    print(f"- {filename} ({len(uploaded_files[filename])} bytes)")

Silakan unggah 5 file Excel Anda:
1. Berkas Pembobot (W8)
2. Berkas Dataset TPT
3. Berkas Dataset Kesehatan
4. Berkas Dataset Pendidikan
5. Berkas Dataset Pertanian


Saving data_JarakCentroid_10th_Pendidikan.xlsx to data_JarakCentroid_10th_Pendidikan.xlsx
Saving data_JarakCentroid_10th_Kesehatan.xlsx to data_JarakCentroid_10th_Kesehatan.xlsx
Saving W8_JarakCentroid_10th_PlusInverseKuantilPlasebo.xlsx to W8_JarakCentroid_10th_PlusInverseKuantilPlasebo.xlsx
Saving data_JarakCentroid_10th_TPT.xlsx to data_JarakCentroid_10th_TPT.xlsx

Berkas yang berhasil diunggah:
- data_JarakCentroid_10th_Pendidikan.xlsx (19182 bytes)
- data_JarakCentroid_10th_Kesehatan.xlsx (20352 bytes)
- W8_JarakCentroid_10th_PlusInverseKuantilPlasebo.xlsx (5317862 bytes)
- data_JarakCentroid_10th_TPT.xlsx (20352 bytes)


### Konfigurasi Nama File dan Analisis Multi-Dataset
Setelah mengunggah berkas-berkas di atas, silakan sesuaikan nama file pada sel di bawah ini sesuai dengan nama file yang Anda unggah, lalu jalankan untuk mengevaluasi seluruh pembobot spasial-temporal terhadap 4 bidang tersebut secara simultan.

In [ ]:
# @title Pemrosesan 3 Bidang Dataset Lintas Seluruh Pembobot W8 (Periode 10 Tahun)
import pandas as pd
import numpy as np
import os
from google.colab import files

# ==========================================
# KONFIGURASI FILE BARU 10 TAHUN
# ==========================================
W8_FILE_PATH = "W8_JarakCentroid_10th_PlusInverseKuantilPlasebo.xlsx"  # File pembobot 10 tahun

# Hanya 3 bidang (tanpa Pertanian) untuk periode 10 tahun
datasets_config = {
    "TPT": "data_JarakCentroid_10th_TPT.xlsx",
    "Kesehatan": "data_JarakCentroid_10th_Kesehatan.xlsx",
    "Pendidikan": "data_JarakCentroid_10th_Pendidikan.xlsx"
}

def fit_eq13_fixed_effect(df_in, Wy):
    """
    Fit: y_it = rho*Wy_it + beta*y_{i,t-1} + mu_i + eps_it
    using within transformation (demean by location).
    """
    dfm = df_in.copy()
    dfm["Wy"] = Wy

    # gunakan hanya baris dengan lag tersedia
    dfm = dfm.dropna(subset=["Y_(t-1)"]).copy()

    y = dfm["Y"].astype(float).to_numpy()
    x1 = dfm["Wy"].astype(float).to_numpy()
    x2 = dfm["Y_(t-1)"].astype(float).to_numpy()
    g  = dfm["loc_id"].to_numpy()

    # within (demean by group)
    y_dm  = y  - pd.Series(y).groupby(g).transform("mean").to_numpy()
    x1_dm = x1 - pd.Series(x1).groupby(g).transform("mean").to_numpy()
    x2_dm = x2 - pd.Series(x2).groupby(g).transform("mean").to_numpy()

    X = np.column_stack([x1_dm, x2_dm])

    # OLS
    XtX = X.T @ X
    Xty = X.T @ y_dm
    coef = np.linalg.solve(XtX, Xty)

    rho_hat, beta_hat = coef[0], coef[1]

    # fitted untuk versi demeaned
    yhat_dm = X @ coef
    resid_dm = y_dm - yhat_dm

    # RMSE dan R2 (within)
    rmse = float(np.sqrt(np.mean(resid_dm**2)))
    ss_res = float(np.sum(resid_dm**2))
    ss_tot = float(np.sum((y_dm - y_dm.mean())**2))
    r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

    return {
        "rho_hat": float(rho_hat),
        "beta_hat": float(beta_hat),
        "rmse_within": rmse,
        "r2_within": r2,
        "n_obs": int(len(dfm))
    }

def compute_Wy_bd_dynamic(df, W_bd):
    """
    W_bd: 35x35, dihitung per tahun secara fleksibel untuk total baris berapapun
    """
    Wy = np.zeros(len(df), dtype=float)
    for yr in sorted(df["Waktu"].unique()):
        idx = df.index[df["Waktu"] == yr].to_numpy()
        y_t = df.loc[idx, "Y"].astype(float).to_numpy().reshape(-1,1)
        if y_t.shape[0] != W_bd.shape[0]:
            raise ValueError(f"Year {yr}: expected {W_bd.shape[0]} rows, got {y_t.shape[0]}")
        Wy[idx] = (W_bd @ y_t).ravel()
    return Wy

# List Lembar Pembobot W_ST dan W_bd
W_ST_sheets = [
    "W_ST^Q_diag1", "W_ST^Q_diag0", "W_ST^Q10", "W_ST^Qq", "W_ST^Q10q",
    "W_ST^Qr", "W_ST^Q10r", "W_ST^ik(ItWs)", "W_bdst", "Wseragam_diag0",
    "Wseragam_diag1", "Wacak_diag0_0001to0999", "Wacak_diag1_0001to0999"
]
W_bd_sheet = "W_bd"

all_results = []

try:
    print("Membaca matriks pembobot dasar dari file Excel...")
    loaded_W_ST_base = {}
    for sh in W_ST_sheets:
        loaded_W_ST_base[sh] = pd.read_excel(W8_FILE_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)
    W_bd = pd.read_excel(W8_FILE_PATH, sheet_name=W_bd_sheet, header=None).to_numpy(dtype=float)

    # Memproses setiap bidang dataset (3 bidang, 10 tahun)
    for nama_bidang, file_path in datasets_config.items():
        if not os.path.exists(file_path):
            print(f"[Peringatan] Berkas untuk bidang '{nama_bidang}' ({file_path}) tidak ditemukan.")
            continue

        print(f"Memproses Bidang: {nama_bidang} ({file_path})...")
        df_bidang = pd.read_excel(file_path)

        # Buat ID lokasi
        loc_order = df_bidang["Lokasi"].drop_duplicates().tolist()
        loc_to_id = {name: i for i, name in enumerate(loc_order)}
        df_bidang["loc_id"] = df_bidang["Lokasi"].map(loc_to_id)

        n_loc = df_bidang["loc_id"].nunique()
        t_years = df_bidang["Waktu"].nunique()
        n_total = len(df_bidang)
        print(f"-> Terdeteksi {n_loc} lokasi x {t_years} tahun = {n_total} baris data.")

        # 1. Jalankan untuk 13 Matriks W_ST (Disesuaikan skalanya ke 10 tahun secara dinamis)
        for sh in W_ST_sheets:
            W_base = loaded_W_ST_base[sh]
            # Memotong ke struktur spasial dasar 35x35 dan mengekspansi menggunakan Kronecker Product sesuai jumlah tahun (10 tahun)
            W_s_base = W_base[:35, :35]
            W_ST_expanded = np.kron(np.eye(t_years), W_s_base)

            Wy = (W_ST_expanded @ df_bidang["Y"].astype(float).to_numpy().reshape(-1, 1)).ravel()
            res = fit_eq13_fixed_effect(df_bidang, Wy)
            res.update({
                "Bidang": nama_bidang,
                "Pembobot": sh,
                "Wy_type": f"W_ST ({n_total}x{n_total})"
            })
            all_results.append(res)

        # 2. Jalankan untuk Matriks W_bd
        Wy_bd = compute_Wy_bd_dynamic(df_bidang, W_bd)
        res_bd = fit_eq13_fixed_effect(df_bidang, Wy_bd)
        res_bd.update({
            "Bidang": nama_bidang,
            "Pembobot": "W_bd",
            "Wy_type": f"W_bd per-year ({n_loc}x{n_loc})"
        })
        all_results.append(res_bd)

    # Gabungkan semua hasil dan tampilkan
    if all_results:
        grand_res_df = pd.DataFrame(all_results)
        cols = ["Bidang", "Pembobot", "Wy_type", "rho_hat", "beta_hat", "r2_within", "rmse_within", "n_obs"]
        grand_res_df = grand_res_df[cols].sort_values(["Bidang", "Pembobot"]).reset_index(drop=True)
        display(grand_res_df)

        # Simpan hasil kompilasi ke Excel
        output_all_fields = "Hasil_Simulasi_Gabungan_3_Bidang_10th.xlsx"
        grand_res_df.to_excel(output_all_fields, index=False)
        print(f"\nHasil kompilasi sukses disimpan ke '{output_all_fields}'!")
        files.download(output_all_fields)
    else:
        print("Tidak ada data yang diproses.")
except Exception as e:
    print(f"Terjadi kesalahan saat memproses data: {e}")

Membaca matriks pembobot dasar dari file Excel...
Memproses Bidang: TPT (data_JarakCentroid_10th_TPT.xlsx)...
-> Terdeteksi 35 lokasi x 10 tahun = 350 baris data.
Memproses Bidang: Kesehatan (data_JarakCentroid_10th_Kesehatan.xlsx)...
-> Terdeteksi 35 lokasi x 10 tahun = 350 baris data.
Memproses Bidang: Pendidikan (data_JarakCentroid_10th_Pendidikan.xlsx)...
-> Terdeteksi 35 lokasi x 10 tahun = 350 baris data.


,Bidang,Pembobot,Wy_type,rho_hat,beta_hat,r2_within,rmse_within,n_obs
0,Kesehatan,W_ST^Q10,W_ST (350x350),0.691691,0.087312,0.883440,1.967687,315
1,Kesehatan,W_ST^Q10q,W_ST (350x350),0.615950,0.119232,0.826824,2.398412,315
2,Kesehatan,W_ST^Q10r,W_ST (350x350),0.639480,0.115999,0.833232,2.353619,315
3,Kesehatan,W_ST^Q_diag0,W_ST (350x350),0.951560,0.213693,0.279548,4.891958,315
4,Kesehatan,W_ST^Q_diag1,W_ST (350x350),0.695794,0.084439,0.887697,1.931415,315
5,Kesehatan,W_ST^Qq,W_ST (350x350),0.615361,0.120978,0.825569,2.407090,315
6,Kesehatan,W_ST^Qr,W_ST (350x350),0.619195,0.128599,0.823070,2.424268,315
7,Kesehatan,W_ST^ik(ItWs),W_ST (350x350),0.937009,0.213774,0.276156,4.903461,315
8,Kesehatan,W_bd,W_bd per-year (35x35),0.848492,0.222935,0.250588,4.989313,315
9,Kesehatan,W_bdst,W_ST (350x350),0.684285,0.097253,0.872004,2.061956,315



Hasil kompilasi sukses disimpan ke 'Hasil_Simulasi_Gabungan_3_Bidang_10th.xlsx'!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#2) Wy building function for W_ST (350×350) and W_bd (35×35)

def compute_Wy_ST(df, W_ST):
    """
    W_ST: 350x350, dikalikan langsung dengan vektor y (panjang 350)
    sesuai urutan baris df.
    """
    y_vec = df["Y"].astype(float).to_numpy().reshape(-1,1)  # 350x1
    Wy = (W_ST @ y_vec).ravel()
    return Wy

def compute_Wy_bd(df, W_bd):
    """
    W_bd: 35x35, dihitung per tahun:
    Wy_t = W_bd @ y_t, lalu distack sesuai urutan df.
    Asumsi: dalam setiap tahun, urutan 35 lokasi di df sama dengan urutan W_bd.
    """
    Wy = np.zeros(len(df), dtype=float)
    for yr in df["Waktu"].unique():
        idx = df.index[df["Waktu"] == yr].to_numpy()
        y_t = df.loc[idx, "Y"].astype(float).to_numpy().reshape(-1,1)  # 35x1
        if y_t.shape[0] != W_bd.shape[0]:
            raise ValueError(f"Year {yr}: expected {W_bd.shape[0]} rows, got {y_t.shape[0]}")
        Wy[idx] = (W_bd @ y_t).ravel()
    return Wy


In [ ]:
#3) Eq.(14) estimation with location fixed effect (within estimator)

def fit_eq13_fixed_effect(df_in, Wy):
    """
    Fit: y_it = rho*Wy_it + beta*y_{i,t-1} + mu_i + eps_it
    using within transformation (demean by location).

    Returns: rho_hat, beta_hat, fitted, residual, R2, RMSE, n_obs
    """
    dfm = df_in.copy()
    dfm["Wy"] = Wy

    # gunakan hanya baris dengan lag tersedia (2019–2022)
    dfm = dfm.dropna(subset=["Y_(t-1)"]).copy()

    y = dfm["Y"].astype(float).to_numpy()
    x1 = dfm["Wy"].astype(float).to_numpy()
    x2 = dfm["Y_(t-1)"].astype(float).to_numpy()
    g  = dfm["loc_id"].to_numpy()

    # within (demean by group)
    y_dm  = y  - pd.Series(y).groupby(g).transform("mean").to_numpy()
    x1_dm = x1 - pd.Series(x1).groupby(g).transform("mean").to_numpy()
    x2_dm = x2 - pd.Series(x2).groupby(g).transform("mean").to_numpy()

    X = np.column_stack([x1_dm, x2_dm])  # tanpa intercept (FE sudah dihapus via demeaning)

    # OLS
    XtX = X.T @ X
    Xty = X.T @ y_dm
    coef = np.linalg.solve(XtX, Xty)

    rho_hat, beta_hat = coef[0], coef[1]

    # fitted untuk versi demeaned
    yhat_dm = X @ coef
    resid_dm = y_dm - yhat_dm

    # RMSE dan R2 (within)
    rmse = float(np.sqrt(np.mean(resid_dm**2)))
    ss_res = float(np.sum(resid_dm**2))
    ss_tot = float(np.sum((y_dm - y_dm.mean())**2))
    r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

    return {
        "rho_hat": float(rho_hat),
        "beta_hat": float(beta_hat),
        "rmse_within": rmse,
        "r2_within": r2,
        "n_obs": int(len(dfm))
    }


In [ ]:
#4) Run for all weights on W8.xlsx across all 3 dataset fields (excluding Pertanian)

import pandas as pd
import numpy as np

W8_PATH = "W8_JarakCentroid_10th_PlusInverseKuantilPlasebo.xlsx"

# Konfigurasi berkas untuk masing-masing bidang
datasets_config = {
    "TPT": "data_JarakCentroid_10th_TPT.xlsx",
    "Kesehatan": "data_JarakCentroid_10th_Kesehatan.xlsx",
    "Pendidikan": "data_JarakCentroid_10th_Pendidikan.xlsx",
}

xls = pd.ExcelFile(W8_PATH)
print("Sheets yang ditemukan di file pembobot:", xls.sheet_names)

W_ST_sheets = ["W_ST^Q_diag1","W_ST^Q_diag0","W_ST^Q10","W_ST^Qq","W_ST^Q10q","W_ST^Qr","W_ST^Q10r","W_ST^ik(ItWs)","W_bdst","Wseragam_diag0","Wseragam_diag1","Wacak_diag0_0001to0999","Wacak_diag1_0001to0999"]
W_bd_sheet  = "W_bd"

# Membaca seluruh matriks pembobot terlebih dahulu
loaded_W_ST = {}
for sh in W_ST_sheets:
    loaded_W_ST[sh] = pd.read_excel(W8_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)
W_bd = pd.read_excel(W8_PATH, sheet_name=W_bd_sheet, header=None).to_numpy(dtype=float)

all_results = []

# Memulai pemrosesan paralel untuk setiap dataset
for nama_bidang, data_file in datasets_config.items():
    print(f"Memproses Bidang: {nama_bidang} ({data_file})...")
    df_bidang = pd.read_excel(data_file)

    # Menyiapkan ID lokasi
    loc_order = df_bidang["Lokasi"].drop_duplicates().tolist()
    loc_to_id = {name: i for i, name in enumerate(loc_order)}
    df_bidang["loc_id"] = df_bidang["Lokasi"].map(loc_to_id)

    n = df_bidang["loc_id"].nunique()

    # 1. Jalankan untuk 13 Matriks W_ST
    for sh in W_ST_sheets:
        W_ST = loaded_W_ST[sh]
        assert W_ST.shape == (len(df_bidang), len(df_bidang)), (sh, W_ST.shape)

        Wy = compute_Wy_ST(df_bidang, W_ST)
        out = fit_eq13_fixed_effect(df_bidang, Wy)
        out["Bidang"] = nama_bidang
        out["Weight"] = sh
        out["Wy_type"] = "W_ST (350x350)"
        all_results.append(out)

    # 2. Jalankan untuk Matriks W_bd
    assert W_bd.shape == (n, n), ("W_bd", W_bd.shape)
    Wy_bd = compute_Wy_bd(df_bidang, W_bd)
    out_bd = fit_eq13_fixed_effect(df_bidang, Wy_bd)
    out_bd["Bidang"] = nama_bidang
    out_bd["Weight"] = "W_bd"
    out_bd["Wy_type"] = "W_bd per-year (35x35)"
    all_results.append(out_bd)

grand_res_df = pd.DataFrame(all_results)
cols = ["Bidang", "Weight", "Wy_type", "rho_hat", "beta_hat", "r2_within", "rmse_within", "n_obs"]
grand_res_df = grand_res_df[cols].sort_values(["Bidang", "Weight"]).reset_index(drop=True)
grand_res_df

Sheets yang ditemukan di file pembobot: ['W_ST^Q_diag1', 'W_ST^Q_diag0', 'W_ST^Q', 'W_ST^Q10', 'W_ST^Qq', 'W_ST^Q10q', 'W_ST^Qr', 'W_ST^Q10r', 'W_ST^ik(ItWs)', 'W_bdst', 'W_bd', 'Wseragam_diag0', 'Wseragam_diag1', 'Wacak_diag0_0001to0999', 'Wacak_diag1_0001to0999']
Memproses Bidang: TPT (data_JarakCentroid_10th_TPT.xlsx)...
Memproses Bidang: Kesehatan (data_JarakCentroid_10th_Kesehatan.xlsx)...
Memproses Bidang: Pendidikan (data_JarakCentroid_10th_Pendidikan.xlsx)...


,Bidang,Weight,Wy_type,rho_hat,beta_hat,r2_within,rmse_within,n_obs
0,Kesehatan,W_ST^Q10,W_ST (350x350),0.691691,-0.604379,0.883440,1.967687,315
1,Kesehatan,W_ST^Q10q,W_ST (350x350),0.395978,-0.244723,0.579486,3.737409,315
2,Kesehatan,W_ST^Q10r,W_ST (350x350),0.639480,-0.523481,0.833232,2.353619,315
3,Kesehatan,W_ST^Q_diag0,W_ST (350x350),0.951560,-0.737867,0.279548,4.891958,315
4,Kesehatan,W_ST^Q_diag1,W_ST (350x350),0.695794,-0.611355,0.887697,1.931415,315
5,Kesehatan,W_ST^Qq,W_ST (350x350),0.615361,-0.494383,0.825569,2.407090,315
6,Kesehatan,W_ST^Qr,W_ST (350x350),0.619195,-0.490596,0.823070,2.424268,315
7,Kesehatan,W_ST^ik(ItWs),W_ST (350x350),0.937009,-0.723235,0.276156,4.903461,315
8,Kesehatan,W_bd,W_bd per-year (35x35),0.848492,0.222935,0.250588,4.989313,315
9,Kesehatan,W_bdst,W_ST (350x350),0.684285,-0.587032,0.872004,2.061956,315
